# Trial Counts by Stage and Mouse

Summary of total trials, free-choice trials, and session counts across
all training stages (4.2–4.7) and experimental sessions (4.8) for 6 mice.

In [ ]:
import sys, os
import numpy as np
import pandas as pd

_d = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isfile(os.path.join(_d, "config.py")):
        break
    _d = os.path.dirname(_d)
sys.path.insert(0, os.path.join(_d, "src"))
sys.path.insert(0, _d)

import data_import as di
from config import SUBJECTS, raw_subject_dir

In [ ]:
STAGES = ['4.2', '4.3', '4.4', '4.5', '4.6', '4.7', '4.8']

records = []

for subject in SUBJECTS:
    # Training sessions (stages 4.2–4.7)
    exp_tr = di.Experiment(raw_subject_dir(subject, training=True))
    for s in exp_tr.get_sessions(subject_IDs='all', when='all'):
        stage = s.training_stage
        if stage in STAGES[:6]:
            lines = s.print_lines
            fc = sum(1 for line in lines if line.split(' ')[9] == 'CT:FC')
            records.append({
                'subject': subject, 'stage': stage,
                'total_trials': len(lines), 'fc_trials': fc
            })

    # Experimental sessions → labeled as stage 4.8
    exp_ex = di.Experiment(raw_subject_dir(subject, training=False))
    for s in exp_ex.get_sessions(subject_IDs='all', when='all'):
        lines = s.print_lines
        fc = sum(1 for line in lines if line.split(' ')[9] == 'CT:FC')
        records.append({
            'subject': subject, 'stage': '4.8',
            'total_trials': len(lines), 'fc_trials': fc
        })

df = pd.DataFrame(records)
print(f"{len(df)} sessions loaded across {df['subject'].nunique()} mice")
print(f"Total trials: {df['total_trials'].sum()}")
df.head(10)

## Total Trials per Stage

In [ ]:
# Pooled summary by stage (all sessions pooled, for reference)
stage_summary = df.groupby('stage')['total_trials'].agg(['count', 'sum', 'mean', 'std']).round(1)
stage_summary.columns = ['n_sessions', 'total', 'pooled_mean', 'pooled_std']

print(f"\nGrand total: {df['total_trials'].sum()} trials")

# Per-mouse mean total trials, then mean +/- SD ACROSS the 6 mice.
# This is the aggregation used in the paper table: each mouse contributes one
# value (its session-mean), and the SD is the spread across mice.
# ddof=0 (population SD, divide by n) reproduces the paper table exactly.
pivot_total = df.groupby(['subject', 'stage'])['total_trials'].mean().unstack()[STAGES]
pivot_total.loc['Mean'] = pivot_total.mean()
pivot_total.loc['Std']  = pivot_total.loc[SUBJECTS].std(ddof=0)   # population SD across mice
pivot_total.round(1)

## Free-Choice Trials per Stage

In [ ]:
# Per-mouse mean FC trials, then mean +/- SD across the 6 mice (ddof=0)
pivot_fc = df.groupby(['subject', 'stage'])['fc_trials'].mean().unstack()[STAGES]
pivot_fc.loc['Mean'] = pivot_fc.mean()
pivot_fc.loc['Std']  = pivot_fc.loc[SUBJECTS].std(ddof=0)
pivot_fc.round(1)

In [ ]:
# ── Paper-table summary: mean ± SD across mice (per-mouse session means) ──
def stage_mean_std(col, ddof=0):
    pm = df.groupby(['subject', 'stage'])[col].mean().unstack()[STAGES]
    return pm.mean(), pm.std(ddof=ddof)

tot_m, tot_s = stage_mean_std('total_trials')
fc_m,  fc_s  = stage_mean_std('fc_trials')

# sessions per mouse (mean across the mice that reached each stage)
sess = df.groupby(['subject', 'stage']).size().unstack()[STAGES]
sess_per_mouse = sess.mean()

summary = pd.DataFrame({
    'total_trials':     [f"{tot_m[s]:.1f} ± {tot_s[s]:.1f}" for s in STAGES],
    'fc_trials':        [f"{fc_m[s]:.1f} ± {fc_s[s]:.1f}"  for s in STAGES],
    'fc_fraction':      [f"{fc_m[s] / tot_m[s]:.3f}"            for s in STAGES],
    'sessions_per_mouse': [f"{sess_per_mouse[s]:.1f}"          for s in STAGES],
}, index=STAGES)
summary.index.name = 'stage'
print("mean ± SD across 6 mice (per-mouse session means; SD is population, ddof=0)")
summary

## Session Counts per Stage

In [ ]:
# Sessions per mouse per stage
pivot_sessions = df.groupby(['subject', 'stage']).size().unstack(fill_value=0)
pivot_sessions = pivot_sessions[STAGES]
pivot_sessions.loc['Total'] = pivot_sessions.sum()
pivot_sessions